# Downstream task: PubMedQA along lambda (Colab, GPU)

Accuracy of the scaled adapters on PubMedQA (all 1,000 expert-labeled questions, PQA-L; yes/no/maybe scored by the log-likelihood of each option, with the prompt and metric of the lm-evaluation-harness task `pubmedqa`), for logit-space and weight-space scaling at every lambda (`scripts/zeroshot_bench.py --tasks pubmedqa_labeled`). The PubMed adapters are the in-domain case, the balanced IMDB adapters the out-of-domain control. The adapters are those already on Drive; nothing is trained. Results go to `/content/drive/MyDrive/dynalora_results/pubmedqa` (one `bench__<model>__<config>.csv.gz` per adapter, with per-question scores only), and every run skips the files that already exist, so the notebook can be re-run after a disconnection.

Runtime: T4 for GPT-2 and Qwen2.5-1.5B (sections 1-2); L4 for Qwen2.5-7B in 4 bits (section 3). After switching the runtime, run the two setup cells again.

In [ ]:
!nvidia-smi -L
!git clone -b paper https://github.com/alexroehrs/dynalora.git /content/dynalora
%cd /content/dynalora
!git pull -q
!pip install -q -r requirements.txt lm-eval bitsandbytes
!pip uninstall -y -q torchao

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/dynalora_results/pubmedqa', exist_ok=True)
print(os.listdir('/content/drive/MyDrive/dynalora_results/pubmedqa'))

## 1. GPT-2 (T4): PubMed adapter and its two seed replicates, balanced IMDB as control, all 13 lambdas

`--max_len 1024` keeps the whole abstract (GPT-2's context length); a few minutes per adapter.

In [ ]:
!python scripts/zeroshot_bench.py --model gpt2 --adapters_dir /content/drive/MyDrive/dynalora_results/gpt2/adapters --configs pubmed imdb_bal --tasks pubmedqa_labeled --max_len 1024 --batch_size 8 --out /content/drive/MyDrive/dynalora_results/pubmedqa
!python scripts/zeroshot_bench.py --model gpt2 --adapters_dir /content/drive/MyDrive/dynalora_results/seeds/adapters --configs pubmed__seed43 pubmed__seed44 --tasks pubmedqa_labeled --max_len 1024 --batch_size 8 --out /content/drive/MyDrive/dynalora_results/pubmedqa

## 2. Qwen2.5-1.5B (T4): PubMed and balanced IMDB, lambda 0, 0.5, ..., 3

In [ ]:
!python scripts/zeroshot_bench.py --model Qwen/Qwen2.5-1.5B --adapters_dir /content/drive/MyDrive/dynalora_results/qwen15/adapters --configs pubmed imdb_bal --tasks pubmedqa_labeled --max_len 1024 --batch_size 4 --lambdas 0 0.5 1 1.5 2 2.5 3 --out /content/drive/MyDrive/dynalora_results/pubmedqa

## 3. Qwen2.5-7B in 4 bits (L4): PubMed and balanced IMDB, lambda 0, 0.5, ..., 3

The base model is loaded in 4-bit NF4, as the adapters were trained against it. Seven forward passes over the 3,000 question-option pairs per adapter; if the full set is too slow, stop the cell, delete the partial output if any, and run the commented line instead (first 500 questions, `--limit 500`, as for the other zero-shot benchmarks of this model).

In [ ]:
!python scripts/zeroshot_bench.py --model Qwen/Qwen2.5-7B --adapters_dir /content/drive/MyDrive/dynalora_results/qwen7b/adapters --configs pubmed imdb_bal --tasks pubmedqa_labeled --load_in_4bit --max_len 1024 --batch_size 4 --lambdas 0 0.5 1 1.5 2 2.5 3 --out /content/drive/MyDrive/dynalora_results/pubmedqa
# !python scripts/zeroshot_bench.py --model Qwen/Qwen2.5-7B --adapters_dir /content/drive/MyDrive/dynalora_results/qwen7b/adapters --configs pubmed imdb_bal --tasks pubmedqa_labeled --load_in_4bit --max_len 1024 --batch_size 4 --limit 500 --lambdas 0 0.5 1 1.5 2 2.5 3 --out /content/drive/MyDrive/dynalora_results/pubmedqa

## 4. Quick look: accuracy along lambda and at equal KL divergence

The same analysis as `scripts/analysis/pubmedqa.py` in the repository, written next to the results.

In [ ]:
!python scripts/analysis/pubmedqa.py --inputs /content/drive/MyDrive/dynalora_results/pubmedqa --out /content/drive/MyDrive/dynalora_results/pubmedqa/analysis

## 5. Package the results for download (scores only, no questions or abstracts)

In [ ]:
!cd /content/drive/MyDrive/dynalora_results && zip -qr /content/dynalora_pubmedqa.zip pubmedqa
from google.colab import files
files.download('/content/dynalora_pubmedqa.zip')